# 08. Cross-Company Comparison — 기업 간 비교

> **Experimental** — 이 튜토리얼의 기능(인사이트 등급, 이상치 탐지, 시장 순위)은 아직 개발 중이다. API와 출력 형식이 변경될 수 있다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/dartlab/blob/master/notebooks/tutorials/08_cross_company.ipynb)

섹터 분류, 인사이트 등급, 시장 순위를 사용하여 여러 기업을 비교한다.

- 섹터 분류 (WICS 11섹터)
- 인사이트 등급 (7영역 A~F)
- 이상치 탐지
- 시장 내 순위
- 종합 비교 테이블

In [ ]:
!pip install -q dartlab

In [ ]:
import dartlab

## 섹터 분류

WICS(WiseFn Industry Classification Standard) 11섹터 체계로 기업을 자동 분류한다.

분류는 3단계로 이루어진다:
1. **오버라이드 매핑** — 수동 지정된 기업
2. **키워드 매칭** — 기업명/사업내용에서 키워드 추출
3. **KSIC 코드 매핑** — 한국표준산업분류 코드 기반

In [ ]:
from dartlab.engines.sector import classify

for code in ["005930", "000660", "035420", "051910"]:
    info = classify(code)
    print(f"{info.corpName}: {info.sector} > {info.industryGroup} (신뢰도: {info.confidence})")

### 섹터별 벤치마크 파라미터

In [ ]:
from dartlab.engines.sector import getParams

params = getParams("005930")
print(f"섹터: {params.sector}")
print(f"할인율: {params.discountRate}%")
print(f"성장률: {params.growthRate}%")
print(f"PER 멀티플: {params.perMultiple}x")

## 인사이트 등급

7개 영역을 A~F로 등급화하여 기업의 강점과 약점을 한눈에 파악한다.

| 영역 | 평가 항목 |
|------|----------|
| performance | 매출/영업이익 성장률 |
| profitability | 영업이익률, ROE |
| health | 부채비율, 유동비율 |
| cashflow | 영업CF, FCF |
| governance | 최대주주 지분, 감사의견 |
| risk | 이상치, 우발부채 |
| opportunity | 성장 포텐셜 |

In [ ]:
from dartlab.engines.insight import analyze

for code in ["005930", "000660", "035420"]:
    result = analyze(code)
    if result is None:
        continue
    grades = result.grades()
    grade_str = " / ".join(f"{k}:{v}" for k, v in grades.items())
    print(f"{result.company.corpName} [{result.profile}]: {grade_str}")

### 이상치 탐지

In [ ]:
result = analyze("005930")
if result:
    for anomaly in result.anomalies:
        print(f"  {anomaly.metric}: Z-score {anomaly.zscore:.1f} ({anomaly.direction})")

## 시장 내 순위

In [ ]:
from dartlab.engines.rank import getRankOrBuild

for code in ["005930", "000660", "035420"]:
    rank = getRankOrBuild(code)
    if rank is None:
        continue
    print(f"\n{rank.corpName} ({rank.sector})")
    print(f"  매출 순위: {rank.revenueRank}/{rank.revenueTotal} (섹터 내 {rank.revenueRankInSector}/{rank.revenueSectorTotal})")
    print(f"  자산 순위: {rank.assetRank}/{rank.assetTotal}")
    print(f"  규모: {rank.sizeClass}")

## 종합 비교 테이블

In [ ]:
import polars as pl
from dartlab.engines.sector import classify

codes = ["005930", "000660", "035420", "051910", "006400"]
rows = []

for code in codes:
    c = dartlab.Company(code)
    r = c.ratios
    sector = classify(code)
    if r is None:
        continue
    rows.append({
        "기업": c.corpName,
        "섹터": sector.sector if sector else "",
        "매출(억)": round(r.revenueTTM / 1e8) if r.revenueTTM else None,
        "영업이익률(%)": round(r.operatingMargin, 1) if r.operatingMargin else None,
        "ROE(%)": round(r.roe, 1) if r.roe else None,
        "부채비율(%)": round(r.debtRatio, 1) if r.debtRatio else None,
        "FCF(억)": round(r.fcf / 1e8) if r.fcf else None,
    })

pl.DataFrame(rows)

---

튜토리얼 완료! 더 자세한 내용은 [API Reference](https://eddmpython.github.io/dartlab/docs/api/overview)를 참조한다.